In [1]:
import dill
import pandas as pd 
import numpy as np
import re
import glob
import os
import json
import xml.etree.ElementTree as ET 
import requests
import io
import time
from dotenv import load_dotenv

load_dotenv()
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY")

np.random.seed(69)

The XML files below are too large for the repo but they are all publicly available at: https://reports.adviserinfo.sec.gov/reports/CompilationReports/IA_INDVL_Feed_03_24_2026.xml.zip

This and other related information can be found at: https://adviserinfo.sec.gov/compilation

To run this properly, download the zip file, then extract all the .xml files to a subdirectory to the same one that contains this notebook named exactly "sec_xml"

In [2]:
xml_files = glob.glob("sec_xml/IA_Indvl_Feeds*.xml")

rows = []

print(f"Found and processed {len(xml_files)} files.")

Found and processed 20 files.


#### Reading the SEC XML files into a data frame

In [3]:
for file in xml_files:
    try:
        tree = ET.parse(file)
        root = tree.getroot()
        
        for indvl in root.findall('Indvls/Indvl'):
            info = indvl.find('Info')
            crd = info.attrib.get('indvlPK')
            first_name = info.attrib.get('firstNm')
            last_name = info.attrib.get('lastNm')
            
            crnt_emps = indvl.find('CrntEmps')
            if crnt_emps is not None:
                for emp in crnt_emps.findall('CrntEmp'):
                    
                    rows.append({
                        'CRD': crd,
                        'First_Name': first_name,
                        'Last_Name': last_name,
                        'Firm_Name': emp.attrib.get('orgNm'),
                        'Street': emp.attrib.get('str1'),
                        'Street2': emp.attrib.get('str2'),
                        'City': emp.attrib.get('city'),
                        'State': emp.attrib.get('st'),
                        'Zip': emp.attrib.get('postlCd')
                    })
                    break
    except Exception as e:
        print(f"Error reading {file}: {e}")
        
df_sec_addresses = pd.DataFrame(rows)
df_sec_addresses['CRD'] = df_sec_addresses['CRD'].astype(str).str.strip()
print(f"Created DataFrame with {len(df_sec_addresses)} rows and {len(df_sec_addresses.columns)} columns.")

Created DataFrame with 431048 rows and 9 columns.


In the following blocks I will create data frames that are representative of the actual information I started with 

I'll use the information from the data frame just created in the above block to create these.

They'll have multiple data columns, some redundant and some unique.

 - One for the data we pulled from the CRM that was used (Zoho)

 - One for the data we pulled from the marketing automation software (ActiveCampaign)

and

 - One for the data that was pulled from the data broker (AdvisorPro)

In [4]:
df_sec_addresses.head(10)

,CRD,First_Name,Last_Name,Firm_Name,Street,Street2,City,State,Zip
0,2367315,GREGORY,MACKO,BUELL INVESTMENT ADVISORS,200 GLASTONBURY BLVD 1ST FL,NaN,GLASTONBURY,None,06033
1,2367317,STEPHEN,SMITH,UBS FINANCIAL SERVICES INC.,1200 HARBOR BOULEVARD,NaN,WEEHAWKEN,None,07086
2,2367342,JAMES,JOURET,MORGAN STANLEY,2000 WESTCHESTER AVENUE,NaN,PURCHASE,None,10577-2530
3,2367346,JOHN,SPADA,MORGAN STANLEY,2000 WESTCHESTER AVENUE,NaN,PURCHASE,None,10577-2530
4,2367354,JOHN,FIORE,JANNEY MONTGOMERY SCOTT LLC,1717 ARCH STREET,NaN,PHILADELPHIA,None,19103
5,2367360,Gershon,Trimpol,"MERRILL LYNCH, PIERCE, FENNER & SMITH INCORPOR...",ONE BRYANT PARK,NaN,NEW YORK,None,10036
6,2367361,LEONARDO,FIORENTINO,J.P. MORGAN SECURITIES LLC,270 PARK AVENUE,NaN,NEW YORK,None,10017
7,2367399,AMY,MCFADDEN,UBS FINANCIAL SERVICES INC.,1200 HARBOR BOULEVARD,NaN,WEEHAWKEN,None,07086
8,2367416,CHARLENE,REYES,CETERA INVESTMENT ADVISERS LLC,1450 AMERICAN LANE,"6TH FLOOR, SUITE 650",SCHAUMBURG,None,60173-2096
9,2367419,AMIN,RATHLE,PNC WEALTH MANAGEMENT LLC,300 FIFTH AVENUE,26TH FLOOR,PITTSBURGH,None,15222-2722


In [5]:
## Function for creating random email addresses from first, last, crd and a random int between 1-4.
## This should allow for an email match rate close to what I had when originally creating this notebook.
## Including the CRD to avoid false matches.
def generate_random_email(first_name, last_name, crd, firm_name):
    random_int = np.random.randint(1, 4)
    email = f"{first_name.lower()}.{last_name.lower()}.{crd}.{random_int}@{firm_name.lower().strip().replace(' ', '').replace(',', '')}.com"
    return email

In [6]:
sample_size        = int(len(df_sec_addresses) * 0.27)

advisorpro_df      = df_sec_addresses.iloc[np.random.choice(len(df_sec_addresses), size=sample_size, replace=False)].reset_index(drop=True)
crm_df             = df_sec_addresses.iloc[np.random.choice(len(df_sec_addresses), size=sample_size, replace=False)].reset_index(drop=True)
active_campaign_df = df_sec_addresses.iloc[np.random.choice(len(df_sec_addresses), size=sample_size, replace=False)].reset_index(drop=True)
print(f"Created advisorpro_df with {len(advisorpro_df)} rows.")
print(f"Created crm_df with {len(crm_df)} rows.")
print(f"Created active_campaign_df with {len(active_campaign_df)} rows.")

Created advisorpro_df with 116382 rows.
Created crm_df with 116382 rows.
Created active_campaign_df with 116382 rows.


In [9]:
advisorpro_df['Email']       = advisorpro_df.apply(lambda row: generate_random_email(row['First_Name'], row['Last_Name'], row['CRD'], row['Firm_Name']), axis=1)
crm_df['Email']              = crm_df.apply(lambda row: generate_random_email(row['First_Name'], row['Last_Name'], row['CRD'], row['Firm_Name']), axis=1)
active_campaign_df['Email']  = active_campaign_df.apply(lambda row: generate_random_email(row['First_Name'], row['Last_Name'], row['CRD'], row['Firm_Name']), axis=1)
df_sec_addresses['Email']    = df_sec_addresses.apply(lambda row: generate_random_email(row['First_Name'], row['Last_Name'], row['CRD'], row['Firm_Name']), axis=1)

print('Head of AdvisorPro Data Frame')
display(advisorpro_df.head(2))
print('Head of CRM Data Frame')
display(crm_df.head(2))
print('Head of Active Campaign Data Frame')
display(active_campaign_df.head(2))
print('Head of SEC Data Frame')
display(df_sec_addresses.head(2))

Head of AdvisorPro Data Frame


,CRD,First_Name,Last_Name,Firm_Name,Street,Street2,City,State,Zip,Email
0,6982992,David,Belding,"AMERIPRISE FINANCIAL SERVICES, LLC",901 3RD AVENUE SOUTH,NaN,MINNEAPOLIS,None,55402,david.belding.6982992.1@ameriprisefinancialser...
1,5905039,PATRICK,IBARRA,USAA INVESTMENT SERVICES COMPANY,9800 FREDERICKSBURG ROAD,NaN,SAN ANTONIO,None,78288-0227,patrick.ibarra.5905039.2@usaainvestmentservice...


Head of CRM Data Frame


,CRD,First_Name,Last_Name,Firm_Name,Street,Street2,City,State,Zip,Email
0,2782733,JEFFREY,BUNCH,CITIGROUP GLOBAL MARKETS INC.,388 GREENWICH STREET,NaN,NEW YORK,None,10013,jeffrey.bunch.2782733.1@citigroupglobalmarkets...
1,4592226,RYAN,ROUILLE,"MML INVESTORS SERVICES, LLC",1295 STATE STREET,NaN,SPRINGFIELD,None,01111-0001,ryan.rouille.4592226.3@mmlinvestorsservicesllc...


Head of Active Campaign Data Frame


,CRD,First_Name,Last_Name,Firm_Name,Street,Street2,City,State,Zip,Email
0,5908212,TYSON,WAKLEY,THE NORDEN GROUP LLC,5255 N EDGEWOOD DRIVE,SUITE 225,PROVO,None,84604,tyson.wakley.5908212.2@thenordengroupllc.com
1,7575411,Dana,McNulty,JFG FAMILY OFFICE,1144 FIFTEENTH STREET,SUITE 3950,DENVER,None,80202,dana.mcnulty.7575411.3@jfgfamilyoffice.com


Head of SEC Data Frame


,CRD,First_Name,Last_Name,Firm_Name,Street,Street2,City,State,Zip,Email
0,2367315,GREGORY,MACKO,BUELL INVESTMENT ADVISORS,200 GLASTONBURY BLVD 1ST FL,NaN,GLASTONBURY,None,06033,gregory.macko.2367315.2@buellinvestmentadvisor...
1,2367317,STEPHEN,SMITH,UBS FINANCIAL SERVICES INC.,1200 HARBOR BOULEVARD,NaN,WEEHAWKEN,None,07086,stephen.smith.2367317.2@ubsfinancialservicesin...


In [ ]:
advisorpro_columns = [
    'CRD',  ### pull from original column
    'First Name', ### pull from original column
    'Last Name', ### pull from original column
    'RIA', 
    'RIA CRD', 
    'Address',  ### Sample from original column
    'City', ### Sample from original column
    'State',  ### Sample from original column
    'Zip',  ### Sample from original column
    'Title', 
    'Phone', 
    'LinkedIn', 
    'Email 1', ### generate with function from above with a 20% NaN rate
    'Email 2', ### generate with function from above with a 80% NaN rate
    'Email 3', ### generate with function from above with a 97% NaN rate
    'Personal Email',  ### generate with function from above with a 40% NaN rate
    'Person Tag - Family',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Hobbies',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Expertise',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Services',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Investments',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Sports Teams',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - School',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Person Tag - Military Status',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Firm Tag - Platform',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Firm Tag - Technology',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Firm Tag - CRM',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Firm AUM',  ### These can all be NaN since they won't be relevant to anything further in this file
    'Firm Total Employees'  ### These can all be NaN since they won't be relevant to anything further in this file
]

crm_columns = [
    'Record Id', ### will just match idx because **shrugs**
    'Contact Name', ### will combine full name from name elements in original df
    'CG Type', ### This I'll randomly fill from a list of CG types as in the original exercise I needed to exclude some CG Types
    'Account Name', ### in the original use this column was a dictionary object containing both "name" and "id" so I'll recreate that with random account id's 
    'First Visit', ### Can be left blank
    'Lead Source', ### Can be left blank
    'Last Activity Time', ### Can be left blank
    'Most Recent Visit', ### Can be left blank
    'First Page Visited', ### Can be left blank
    'Survey Name' ### Can be left blank
]

active_campaign_columns = [
    'List Name', ### This will be treated similar to the CG type column above
    'Status', ### NaN
    '*Position', ### NaN
    '*Relationship Type', ### NaN
    '*Current CRM Provider', ### NaN
    '*Source', ### NaN
    '*Zip / Postal Code', ### Will pull from original df
    '*Company', ### Will pull from original df
    '*Secondary Email', ### Will fill with a NaN rate of ~60%
    '*CRD' ### Will pull from original df
]

def email_with_nan(df, nan_rate):
    emails = df.apply(
        lambda r: generate_random_email(
            r.get('First Name', r.get('First_Name', 'x')),
            r.get('Last Name', r.get('Last_name', 'x')),
            str(r['CRD']),
            r.get('RIA', r.get('Firm_Name', 'firm'))
        ), axis = 1
    )
    mask = np.random.random(len(df)) < nan_rate 
    emails[mask] = np.nan
    return emails

advisorpro_df['Email 1'] = email_with_nan(advisorpro_df, 0.20)
advisorpro_df['Email 2'] = email_with_nan(advisorpro_df, 0.80)
advisorpro_df['Email 3'] = email_with_nan(advisorpro_df, 0.97)
advisorpro_df['Personal Email'] = email_with_nan(advisorpro_df, 0.40)

crm_df['Record Id'] = crm_df.index
crm_df['Contact Name'] = crm_df['First_Name'] + ' ' + crm_df['Last_Name']
cg_types = ['Bronze', 'Silver', 'Off', 'Prospect', 'PERM OFF', 'Retired']
crm_df['CG Type'] = np.random.choice(cg_types, size=len(crm_df))
crm_df['Account Name'] = crm_df.apply(
    lambda r: {'name': r['Firm_Name'], 'id': np.random.randint(10000, 99999)}, axis = 1
)

advisorpro_df = advisorpro_df.rename(columns = {
    'First_Name': 'First Name',
    'Last_Name': 'Last Name',
    'Firm_Name': 'RIA',
    'Street': 'Address'
})

list_names = ['eGor', 'Prospects', 'Offs', 'Client Team Member']
active_campaign_df['List Name'] = np.random.choice(list_names, size = len(active_campaign_df))
active_campaign_df['*CRD'] = active_campaign_df['CRD']
active_campaign_df['*Company'] = active_campaign_df['Firm_Name']
active_campaign_df['*Zip / Postal Code'] = active_campaign_df['Zip']
active_campaign_df['*Secondary Email'] = email_with_nan(active_campaign_df, 0.31)

for col in advisorpro_columns:
    if col not in advisorpro_df.columns:
        advisorpro_df[col] = np.nan

for col in crm_columns:
    if col not in crm_df.columns:
        crm_df[col] = np.nan

for col in active_campaign_columns:
    if col not in active_campaign_df.columns:
        active_campaign_df[col] = np.nan

for df in [advisorpro_df, crm_df, active_campaign_df]:
    pct_to_remove = np.random.uniform(0.1, 0.5)
    row_to_wipe = np.random.choice(len(df), size = int(len(df) * pct_to_remove), replace=False)
    df.loc[df.index[row_to_wipe], 'Email'] = np.nan

print(f"AdvisorPro: {len(advisorpro_df)} rows, Columns {advisorpro_df.columns}")
print(f"ActiveCampaign: {len(active_campaign_df)} rows, Columns {active_campaign_df.columns}")
print(f"Zoho: {len(crm_df)} rows, Columns {crm_df.columns}")

In [ ]:
advisorpro_df['full_address'] = (
    advisorpro_df['Address'].fillna('') +
    advisorpro_df['Street2'].fillna('') + ", " +
    advisorpro_df['City'].fillna('') + ", " +
    advisorpro_df['State'].fillna('') + ", " +
    advisorpro_df['Zip'].fillna('').astype(str)
    )

crm_df['full_address'] = (
    crm_df['Street'].fillna('') +
    crm_df['Street2'].fillna('') + ", " +
    crm_df['City'].fillna('') + ", " +
    crm_df['State'].fillna('') + ", " +
    crm_df['Zip'].fillna('').astype(str)
    )

active_campaign_df['full_address'] = (
    active_campaign_df['Street'].fillna('') +
    active_campaign_df['Street2'].fillna('') + ", " +
    active_campaign_df['City'].fillna('') + ", " +
    active_campaign_df['State'].fillna('') + ", " +
    active_campaign_df['Zip'].fillna('').astype(str)
    )

active_campaign_df['full_address'].head(10)

In [ ]:
def df_merge_prep(df, source_name):
    out = pd.DataFrame()
    out['CRD'] = df['CRD'].astype(str).str.strip()
    
    if 'First Name' in df.columns:
        out['First_Name'] = df['First Name']
        out['Last_Name'] = df['Last Name']
    elif 'Contact Name' in df.columns:
        split = df['Contact Name'].str.split(' ', n=1, expand = True)
        out['First_Name'] = split[0]
        out['Last_Name'] = split[1] if 1 in split.columns else np.nan
    else:
        out['First_Name'] = df.get('First_Name')
        out['Last_Name'] = df.get('Last_Name')

    for col in ['RIA', 'Firm_Name', '*Company', 'Account (Previously Organization)']:
        if col in df.columns:
            out['Company'] = df[col]
            break
    
    for col in ['Address', 'Street', 'Street2', '*Address']:
        if col in df.columns:
            out['*Address'] = df[col]
            break
    out['*Address2'] = df.get('Street2')

    for col in ['City', '*City']:
        if col in df.columns:
            out['*City'] = df[col]
            break
    for col in ['State', '*State']:
        if col in df.columns:
            out['*State'] = df[col]
            break
    for col in ['Zip', '*Zip / Postal Code']:
        if col in df.columns:
            out['*Zip / Postal Code'] = df[col]
            
    out['*Country'] = 'US' 
    ##-----^^^^ This was originally a regex-based operation for assuming country
    ##-----^^^^ Based on the presence of alphanumeric characters (the original datasets only contained either US or Canadian based Financial Advisors)
    
    email_cols = [c for c in df.columns if 'email' in c.lower()]
    out['Email'] = df[email_cols].apply(
        lambda row: next((v for v in row if pd.notna(v) and str(v).strip()), np.nan),
        axis = 1
    ) if email_cols else np.nan
    
    
    for ec in email_cols:
        safe_name = f'email_{source_name}_{ec}'.replace(' ', '_').replace('*', '')
        out[safe_name] = df[ec]
        
    out['full_address'] = df.get('full_address')
    out['_source'] = source_name
    return out
    
prepped_ap = df_merge_prep(advisorpro_df, 'advisorpro')
prepped_crm = df_merge_prep(crm_df, 'crm')
prepped_ac = df_merge_prep(active_campaign_df, 'activecampagin')

combined = pd.concat([prepped_ap, prepped_crm, prepped_ac], ignore_index = True)
print(f"Combined: {len(combined)} rows")

final_df = combined.groupby('CRD', sort=False).first()
final_df = final_df.reset_index()

final_df['Account (previously Organization)'] = final_df['Company']

In [ ]:
URL = "https://places.googleapis.com/v1/places:searchText"

HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
    "X-Goog-FieldMask": "places.formattedAddress,places.addressComponents"
}

def parse_address_components(components):
    result = {
        "street": None, "city": None, "state": None, "postal_code": None, "country": None
    }
    street_number = None
    route = None
    for comp in components:
        types = comp.get("types", [])
        long_text = comp.get("longText") or comp.get("shortText")
        short_text = comp.get("shortText") or comp.get("longText")

        if "street_number" in types:
            street_number = long_text
        elif "route" in types:
            route = long_text
        elif "locality" in types:
            result["city"] = long_text
        elif "postal_town" in types and result["city"] is None:
            result["city"] = long_text
        elif "administrative_area_level_1" in types:
            result["state"] = short_text
        elif "postal_code" in types:
            result["postal_code"] = long_text
        elif "country" in types:
            result["country"] = short_text

        if route:
            result["street"] = f"{street_number} {route}" if street_number else route
        return result
    
def get_google_address(text_query):
    if not isinstance(text_query, str) or not text_query.strip():
        return None
    data = {"textQuery": text_query}
    try:
        resp = requests.post(URL, json=data, headers = HEADERS, timeout = 10)
        if resp.status_code != 200:
            print(f"Error1 {resp.status_code} - {resp.text[:200]}")
            return None
        places = resp.json().get("places", [])
        if not places:
            return None
        place = places[0]
        parsed = parse_address_components(place.get("addressComponents", []))
        parsed["formatted_address"] = place.get("formattedAddress")
        return parsed
    except Exception as e:
        print(f"Error2: {e}")
        return None

In [ ]:
STREET_COLS = [
    "*Address", "Mailing Street", "Mailing Street_zoho_new",
    "Street", "New_Street"
]
CITY_COLS = [
    "*City", "City", "Mailing City", "Mailing City_zoho_new",
    "City_ap", "City_ap_new"
]
STATE_COLS = [
    "*State", "State", "Mailing State", "Mailing State_zoho_new"
]
ZIP_COLS = [
    "*Zip / Postal Code", "Zip", "Mailing Zip", "Mailing Zip_zoho_new"
]
COUNTRY_COLS = ["*Country"]
COMPANY_COLS = [
    "Firm_Name", "Account (previously Organization)", "Organization", "*Company"
]

def first_not_na(row, cols):
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            return str(row[c]).strip()
    return None

def build_query(row):
    parts = []
    for col_group in [COMPANY_COLS, STREET_COLS, CITY_COLS, STATE_COLS, ZIP_COLS, COUNTRY_COLS]:
        val = first_not_na(row, col_group)
        if val:
            parts.append(val)
    return ", ".join(parts)

mask = (
    final_df['*Address'].notna() &
    (
        final_df['*City'].isna() |
        final_df['*State'].isna() |
        final_df['*Zip / Postal Code'].isna()
    )
)

subset_to_fix = final_df[mask].copy()
subset_to_fix = subset_to_fix.sample(frac=0.01, random_state=93).copy()
unique_queries = subset_to_fix.apply(build_query, axis=1).nunique()

print(f"Unique queries to enrich: {unique_queries}. Rows being attempted: {len(subset_to_fix)}")

## Here is where we actually start building the cache and hitting the API

Cleaning the full data frame we've made using the API takes about 40 minutes so we'll reduce the size of the data frame to about 10% of its size

In [ ]:
cache = {}

cache_path = 'google_address_cache.json'
if os.path.exists(cache_path):
    with open(cache_path) as f:
        cache = json.load(f)
else:
    cache = {}

for i, (idx, row) in enumerate(subset_to_fix.iterrows()):
    if i % 1 == 0:
        print(f"  {i}/{len(subset_to_fix)} done, cache: {len(cache)}", end='\r')
    query = build_query(row)
    if not query:
        continue
    if query in cache:
        addr_data = cache[query]
    else:
        addr_data = get_google_address(query)
        cache[query] = addr_data
        # time.sleep(0.02)
    
    if not addr_data:
        continue
    
    if pd.isna(final_df.at[idx, '*City']) and addr_data.get('city'):
        final_df.at[idx, '*City'] = addr_data['city']
        
    if pd.isna(final_df.at[idx, '*State']) and addr_data.get('state'):
        final_df.at[idx, '*State'] = addr_data['state']
        
    if pd.isna(final_df.at[idx, '*Zip / Postal Code']) and addr_data.get('postal_code'):
        final_df.at[idx, '*Zip / Postal Code'] = addr_data['postal_code']
        
    if pd.isna(final_df.at[idx, '*Country']) and addr_data.get('country'):
        final_df.at[idx, '*Country'] = addr_data['country']
        
    if pd.isna(final_df.at[idx, '*Address']) and addr_data.get('street'):
        final_df.at[idx, '*Address'] = addr_data['street']
        
    if "Full Address" in final_df.columns:
        if pd.isna(final_df.at[idx, "Full Address"]) and addr_data.get('formatted_address'):
            final_df.at[idx, 'Full Address'] = addr_data['formatted_address']
            
print(f'First pass complete. Cache size: {len(cache)} unique addresses.')

with open(cache_path, 'w') as f:
    json.dump(cache, f)

address_missing_mask = (
    final_df['*Address'].isna() &
    final_df['Account (previously Organization)'].notna() &
    final_df['*City'].notna()
)

address_missing_subset = subset_to_fix[address_missing_mask].copy()
print(f'Second Pass: trying to resolve {len(address_missing_subset)} missing street addresses')

for i, (idx, row) in enumerate(address_missing_subset.iterrows()):
    if i % 1 == 0:
        print(f"  {i}/{len(address_missing_subset)} done, cache: {len(cache)}", end='\r')
    query = ' '.join(filter(None, [
        str(row['Account (previously Organization)']),
        str(row['*City']),
        str(row['*State']) if pd.notna(row.get('*State')) else ''
    ]))
    
    if query in cache:
        addr_data = cache[query]
    else:
        addr_data = get_google_address(query)
        cache[query] = addr_data
        # time.sleep(0.02)
        
    if addr_data and addr_data.get('street'):
        final_df.at[idx, '*Address'] = addr_data['street']
        
print(f'Second Pass finished: Rows still missing address: {final_df['*Address'].isna().sum()}')

In [ ]:
# with open(cache_path, 'w') as f:
#     json.dump(cache, f)

# address_missing_mask = (
#     final_df['*Address'].isna() &
#     final_df['Account (previously Organization)'].notna() &
#     final_df['*City'].notna()
# )

# address_missing_subset = final_df[address_missing_mask].copy()
# print(f'Second Pass: trying to resolve {len(address_missing_subset)} missing street addresses')

# for idx, row in address_missing_subset.iterrows():
#     query = ' '.join(filter(None, [
#         str(row['Account (previously Organization)']),
#         str(row['*City']),
#         str(row['*State']) if pd.notna(row.get('*State')) else ''
#     ]))
    
#     if query in cache:
#         addr_data = cache[query]
#     else:
#         addr_data = get_google_address(query)
#         cache[query] = addr_data
#         # time.sleep(0.02)
        
#     if addr_data and addr_data.get('street'):
#         final_df.at[idx, '*Address'] = addr_data['street']
        
# print(f'Second Pass finished: Rows still missing address: {final_df['*Address'].isna().sum()}')

In [ ]:
final_df.shape

In [ ]:
direct_mail_columns = [
    'CRD', 'First_Name', 'Last_Name', 'Company',
    '*Address', '*Address2', '*City', '*State',
    '*Zip / Postal Code', '*Country'
]

available_direct_mail_columns = [c for c in direct_mail_columns if c in final_df.columns]

print(available_direct_mail_columns)

direct_mail_df = final_df[available_direct_mail_columns].copy()

print("direct   mail    df  1")
display(direct_mail_df.head())

direct_mail_df = direct_mail_df = direct_mail_df.dropna(subset = ['*Address', '*City'], how='any')
print("direct   mail    df  after   drop    subset")
display(direct_mail_df.head())
direct_mail_df = direct_mail_df.drop_duplicates(subset=['CRD'])
print("direct   mail    df  after   drop    duplicates")
display(direct_mail_df.head())

direct_mail_df.to_csv('direct_mail_list.csv', index = False)
print(f'Finished direct mail list. Size: {len(direct_mail_df)}')

email_source_columns = [c for c in final_df.columns if 'email' in c.lower()]
id_columns = ['CRD', 'First_Name', 'Last_Name', 'Company']
id_columns_available = [c for c in id_columns if c in final_df.columns]

email_list_long = final_df[id_columns_available + email_source_columns].melt(
    id_vars = id_columns_available,
    value_vars = email_source_columns,
    var_name = 'email_source',
    value_name = 'email_address'
)

email_list_long = email_list_long.dropna(subset = ['email_address'])
email_list_long = email_list_long[email_list_long['email_address'].str.strip() != '']
email_list_long = email_list_long.drop_duplicates(subset = ['CRD', 'email_address'])
email_list_long = email_list_long.sort_values(['CRD', 'email_source']).reset_index(drop = True)

email_list_long.to_csv('email_marketing_list.csv', index = False)

print('Sample of direct mail list:')
display(direct_mail_df.head())
print('Sample of email marketing list:')
display(email_list_long.head())

In [ ]:
email_list_long.shape

In [ ]:
URL = "https://places.googleapis.comm/v1/places:searchText"
def get_google_address(company, city, state):
    if pd.isna(company) or pd.isna(city):
        return None
    query = f"{company} {city} {state if pd.notna(state) else ''}"
    headers = {
       "Content-Type": "application/json",
       "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
       "X-Goog-FieldMask": "places.formattedAddress"
   } 
    data = {"textQuery": query}
    
    try:
        response = requests.post(URL, json=data, headers=headers)
        if response.status_code == 200:
            results = response.json().get('places', [])
            if results:
                return results[0]['formattedAddress']
    except Exception as e:
        print(f"Error: {e}")

In [ ]:
HEADERS = {
    "Content-Type": "application/json",
    "X-Goog-Api-Key": GOOGLE_PLACES_API_KEY,
    "X-Goog-FieldMask": "places.formattedAddress,places.addressComponents"
}

def _parse_address_components(components):
    result = {
        "street": None,
        "city": None,
        "state": None,
        "postal_code": None,
        "country": None
    }
    street_number = None
    route = None
    for comp in components:
        types = comp.get("types", [])
        long_text = comp.get("longText") or comp.get("shortText")
        short_text = comp.get("shortText") or comp.get("longText")

        if "street_number" in types:
            street_number = long_text
        elif "route" in types:
            route = long_text
        elif "locality" in types:
            result["city"] = long_text
        elif "postal_town" in types and result["city"] is None:
            result["city"] = long_text
        elif "administrative_area_level_1" in types:
            result["state"] = short_text
        elif "postal_code" in types:
            result["postal_code"] = long_text
        elif "country" in types:
            result["country"] = short_text

    if route:
        result["street"] = f"{street_number} {route}" if street_number else route
    return result

def get_google_address_components(text_query):
    if not isinstance(text_query, str) or not text_query.strip():
        return None
    
    data = {"textQuery": text_query}
    try:
        resp = requests.post(URL, json = data, headers = HEADERS, timeout=10 )
        if resp.status_code != 200:
            print(f"Google Places error {resp.status_code} - {resp.text[:200]}")
            return None
        places = resp.json().get("places", [])
        if not places:
            return None
        place = places[0]
        formatted = place.get("formattedAddress")
        components = place.get("addressComponents", [])
        parsed = _parse_address_components(components)
        parsed["formatted_address"] = formatted
        return parsed
    except Exception as e:
        print(f"Error with Google API call: {e}")
        return None
    

STREET_COLS = [
    "*Address", "Mailing Street", "Mailing Street_zoho_new",
    "Street", "New_Street"
]
CITY_COLS = [
    "*City", "City", "Mailing City", "Mailing City_zoho_new",
    "City_ap", "City_ap_new"
]
STATE_COLS = [
    "*State", "State", "Mailing State", "Mailing State_zoho_new"
]
ZIP_COLS = [
    "*Zip / Postal Code", "Zip", "Mailing Zip", "Mailing Zip_zoho_new"
]
COUNTRY_COLS = [
    "*Country"
]
COMPANY_COLS = [
    "Firm_Name", "Account (previously Organization)", "Organization", "*Company"
]    

def _first_notna(row, cols):
    for c in cols:
        if c in row.index and pd.notna(row[c]):
            return str(row[c]).strip()
    return None

def build_text_query_from_row(row):
    parts = []
    company = _first_notna(row, COMPANY_COLS)
    street = _first_notna(row, STREET_COLS)
    city = _first_notna(row, CITY_COLS)
    state = _first_notna(row, STATE_COLS)
    zipc = _first_notna(row, ZIP_COLS)
    country = _first_notna(row, COUNTRY_COLS)
    
    for part in [company, street, city, state, zipc, country]:
        if part:
            parts.append(part)
    return ", ".join(parts)

mask = (
    final_df["*Address"].notna() &
    {
        final_df["*City"].isna() |
        final_df["*State"].isna() |
        final_df["*Zip / Postal Code"].isna() |
        final_df["*Country"].isna()
    }
)

subset_to_fix = final_df[mask].copy()
print(f"Attemptin to enrich {len(subset_to_fix)} addresses from Google...")

### Since many firms have multiple FA's we'll build a cache for repeat addresses
### this way we don't send too many queries to the API (The search text endpoint maxes at 75k requests/day)
### I'll probably add something that shows the number of queries that will be sent before actually calling the API
### Another good reason to build the cache before we actually call the API

cache = {}

for idx, row in subset_to_fix.iterrows():
    query = build_text_query_from_row(row)
    if not query:
        continue
    if query in cache:
        addr_data = cache[query]
    else:
        addr_data = get_google_address_components(query)
        cache[query] = addr_data
    
    if not addr_data:
        continue
    
    if pd.isna(final_df.at[idx, "*City"]) and addr_data.get("city"):
        final_df.at[idx, "*City"] = addr_data["city"]
    if pd.isna(final_df.at[idx, "*State"]) and addr_data.get("state"):
        final_df.at[idx, "*State"] = addr_data["state"]

    if pd.isna(final_df.at[idx, "*Zip / Postal Code"]) and addr_data.get("postal_code"):
        final_df.at[idx, "*Zip / Postal Code"] = addr_data["postal_code"]

    if pd.isna(final_df.at[idx, "*Country"]) and addr_data.get("country"):
        final_df.at[idx, "*Country"] = addr_data["country"]

    if pd.isna(final_df.at[idx, "*Address"]) and addr_data.get("street"):
        final_df.at[idx, "*Address"] = addr_data["street"]

    if "Full Address" in final_df.columns:
        if pd.isna(final_df.at[idx, "Full Address"]) and addr_data.get("formatted_address"):
            final_df.at[idx, "Full Address"] = addr_data["formatted_address"]
            
mask_addr_missing = (
    final_df["*Address"].isna() &
    final_df["Account (previously Organization)"].notna() &
    final_df["*City"].notna()
)

subset_addr_missing = final_df[mask_addr_missing].copy()

for idx, row in subset_addr_missing.iterrows():
    query = ' '.join([
        str(row['Account (previously Organization)']),
        str(row['*City']),
        str(row['*State']) if pd.notna(row['*State']) else ''
    ])
    
    
    addr_data = get_google_address(query)
    if addr_data and addr_data.get('street'):
        final_df.at[idx, "*Address"] = addr_data['street']
        time.sleep(0.1)
    
print(f'Enrichment complete. Remaining null addresses: {final_df['*Address'].isna().sum}')

In [ ]:
direct_mail_cols = [
    'CRD', 'First_Name', 'Last_Name', 'Company',
    '*Address', '*Address2', '*City', '*State',
    'Zip / Postal Code', '*Country'
]